# Build DB-COMP Manifest

This notebook builds the DB-COMP source/document manifest and the DB-COMP download manifest.

It creates:

1. `output/com_db_comp/db_comp_manifest.csv` — what was found on DB-COMP, with normalized case numbers and completed DB-COMP document IDs.
2. `output/com_db_comp/db_comp_download_manifest.csv` — download targets and download status.

Raw data is stored in:

- `data/raw/com_db_comp/html/page_0001.html`
- `data/raw/com_db_comp/files/...`

The downloader checks the expected source/download path first. If the file is already present, it records `already_downloaded` instead of downloading again.


### DMA exclusion

This notebook deliberately excludes DB-COMP rows/documents that appear to concern the Digital Markets Act (DMA), because DMA material is outside the Article 101/102/106 antitrust dataset scope. The exclusion is applied before saving `db_comp_manifest.csv` and before building `db_comp_download_manifest.csv`, so DMA PDFs should not be downloaded. Matching is case-insensitive and checks source text, document names, PDF/file names, URLs, and document IDs for `DMA`, `Digital Markets Act`, and `Regulation (EU) 2022/1925` variants.


### Stable leading ID columns

Both output manifests put the two main tracking columns first:

- `dbcomp_document_id`
- `case_number`

Missing `dbcomp_document_id` values are filled deterministically after sorting by `file_url`, starting at `9000`.


In [1]:
import os
import re
import json
import time
import hashlib
import logging
import random
from datetime import datetime
from urllib.parse import urlencode, urljoin, urlparse, unquote

import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm


## 1. Project paths

In [2]:
# Current working directory of the notebook
NOTEBOOK_DIR = os.getcwd()

# Go two levels up: code/notebooks -> code -> project root
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, "..", ".."))

CODE_DIR = os.path.join(PROJECT_ROOT, "code")
NOTEBOOKS_DIR = os.path.join(CODE_DIR, "notebooks")
SCRIPTS_DIR = os.path.join(CODE_DIR, "scripts")
SRC_DIR = os.path.join(CODE_DIR, "src")

CONFIG_DIR = os.path.join(PROJECT_ROOT, "config")
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "output")
LOGS_DIR = os.path.join(PROJECT_ROOT, "logs")

RAW_DATA_DIR = os.path.join(DATA_DIR, "raw")
COM_DB_COMP_RAW_DIR = os.path.join(RAW_DATA_DIR, "com_db_comp")
COM_DB_COMP_HTML_DIR = os.path.join(COM_DB_COMP_RAW_DIR, "html")
COM_DB_COMP_FILES_DIR = os.path.join(COM_DB_COMP_RAW_DIR, "files")
COM_DB_COMP_DEBUG_DIR = os.path.join(COM_DB_COMP_RAW_DIR, "debug")

COM_DB_COMP_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "com_db_comp")
COM_DB_COMP_LOGS_DIR = os.path.join(LOGS_DIR, "com_db_comp")

for path in [
    COM_DB_COMP_RAW_DIR,
    COM_DB_COMP_HTML_DIR,
    COM_DB_COMP_FILES_DIR,
    COM_DB_COMP_DEBUG_DIR,
    COM_DB_COMP_OUTPUT_DIR,
    COM_DB_COMP_LOGS_DIR,
]:
    os.makedirs(path, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Raw data:", COM_DB_COMP_RAW_DIR)
print("Output:", COM_DB_COMP_OUTPUT_DIR)
print("Logs:", COM_DB_COMP_LOGS_DIR)


PROJECT_ROOT: /home/edik/projects/eccjeu
Raw data: /home/edik/projects/eccjeu/data/raw/com_db_comp
Output: /home/edik/projects/eccjeu/output/com_db_comp
Logs: /home/edik/projects/eccjeu/logs/com_db_comp


## 2. Configuration

In [3]:
BASE_URL = "https://db-comp.eu/"

# Search parameters from the original build_dbcomp_manifest notebook.
# These correspond to details search, document types 1/5/2, Jan 1964 to Dec 2026.
SEARCH_CONFIG = {
    "slotName": "Slot_Main_3",
    "Search_by_value": "details",
    "Document_type_values": "1,5,2",
    "Month_from_value": "1",
    "Year_from_value": "1964",
    "Month_to_value": "12",
    "Year_to_value": "2026",
    "Mysql_operator_value": "none",
    "Sort_results_value": "relevance",
}

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 Academic research scraper for non-commercial "
        "EC competition-law dataset validation"
    ),
    "Accept-Language": "en-US,en;q=0.9,de;q=0.8",
}

# Scraping behavior
REBUILD_WEB_SCRAPE_MANIFEST = False   # False = preserve/load existing manifest if present
OVERWRITE_RAW_HTML = False            # False = reuse saved page HTML if present
PAGE_LIMIT = None                     # e.g. 5 for testing; None for all pages
FALLBACK_LAST_PAGE = 44               # used if last page cannot be parsed
REQUEST_SLEEP_SECONDS = (1.0, 2.5)    # polite random sleep between page fetches

# Scope filtering
EXCLUDE_DMA_DOCUMENTS = True          # True = remove Digital Markets Act / DMA documents from manifests and downloads

# Download behavior
DOWNLOAD_FILES = False                 # True = download missing files; existing files get status already_downloaded
OVERWRITE_EXISTING_FILES = False
DOWNLOAD_LIMIT = None                 # e.g. 20 for testing; None for all pending files
SAVE_EVERY = 10
DOWNLOAD_SLEEP_SECONDS = (0.3, 0.8)
DOWNLOAD_TIMEOUT_SECONDS = 30

DB_COMP_MANIFEST_CSV = os.path.join(COM_DB_COMP_OUTPUT_DIR, "db_comp_manifest.csv")
DB_COMP_MANIFEST_PARQUET = os.path.join(COM_DB_COMP_OUTPUT_DIR, "db_comp_manifest.parquet")

# Backward-compatible input name from older notebook versions.
LEGACY_WEB_SCRAPE_MANIFEST_CSV = os.path.join(COM_DB_COMP_OUTPUT_DIR, "web_scrape_manifest.csv")
LEGACY_WEB_SCRAPE_MANIFEST_PARQUET = os.path.join(COM_DB_COMP_OUTPUT_DIR, "web_scrape_manifest.parquet")

# Internal aliases used by helper functions below.
WEB_SCRAPE_MANIFEST_CSV = DB_COMP_MANIFEST_CSV
WEB_SCRAPE_MANIFEST_PARQUET = DB_COMP_MANIFEST_PARQUET
DOWNLOAD_MANIFEST_CSV = os.path.join(COM_DB_COMP_OUTPUT_DIR, "db_comp_download_manifest.csv")
DOWNLOAD_MANIFEST_PARQUET = os.path.join(COM_DB_COMP_OUTPUT_DIR, "db_comp_download_manifest.parquet")

# Backward-compatible input name from older notebook versions.
LEGACY_DOWNLOAD_MANIFEST_CSV = os.path.join(COM_DB_COMP_OUTPUT_DIR, "download_manifest.csv")
LEGACY_DOWNLOAD_MANIFEST_PARQUET = os.path.join(COM_DB_COMP_OUTPUT_DIR, "download_manifest.parquet")


## 3. Logging

In [4]:
log_file = os.path.join(
    COM_DB_COMP_LOGS_DIR,
    f"com_db_comp_scraper_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log",
)

logger = logging.getLogger("com_db_comp_scraper")
logger.setLevel(logging.INFO)
logger.handlers.clear()

file_handler = logging.FileHandler(log_file, encoding="utf-8")
file_handler.setLevel(logging.INFO)
file_handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(file_handler)

# Keep notebook output clean; detailed output goes to the log file.
logger.propagate = False

print("Log file:", log_file)


Log file: /home/edik/projects/eccjeu/logs/com_db_comp/com_db_comp_scraper_20260703_154708.log


## 4. Utility functions

In [5]:
def clean_text(value):
    if value is None:
        return ""
    return re.sub(r"\s+", " ", str(value).replace(" ", " ")).strip()


def make_page_url(page: int) -> str:
    params = dict(SEARCH_CONFIG)
    params["page"] = str(page)
    return BASE_URL + "?" + urlencode(params)


def safe_filename(value, default="file"):
    value = clean_text(value)
    value = unquote(value)
    value = re.sub(r"[^A-Za-z0-9._-]+", "_", value).strip("._-")
    return value or default


def hash_text(value, n=12):
    return hashlib.sha1(str(value).encode("utf-8")).hexdigest()[:n]


def extract_filename_from_url(url):
    if not url:
        return ""
    path = urlparse(url).path
    name = os.path.basename(path)
    return safe_filename(name, default="")


def sleep_random(bounds):
    if bounds is None:
        return
    low, high = bounds
    if high > 0:
        time.sleep(random.uniform(low, high))


## 5. Save/load helpers

## 5a. Scope filter: exclude DMA / Digital Markets Act material

DB-COMP can contain PDF rows relating to the Digital Markets Act (DMA). These are not competition-law Commission antitrust decisions for the current Article 101/102/106 dataset, so the notebook removes them from both the DB-COMP manifest and the download manifest.

The filter is intentionally conservative but broad enough to catch normal DB-COMP naming patterns: it checks `document_name`, `pdf_filename`, `file_name`, `file_url`, DB-COMP document URLs/IDs, and raw result text for DMA / Digital Markets Act / Regulation (EU) 2022/1925 wording.


In [6]:
WEB_SCRAPE_COLUMNS = [
    "dbcomp_document_id",
    "case_number",
    "source",
    "page",
    "page_url",
    "result_index",
    "db_comp_case_number_saved",
    "db_comp_case_number_extracted_from_text",
    "document_name",
    "decision_date",
    "pdf_filename",
    "dbcomp_document_url",
    "file_url",
    "file_name",
    "file_extension",
    "raw_links_json",
    "raw_result_text",
    "scrape_status",
    "scrape_error",
]

DOWNLOAD_RESULT_COLUMNS = [
    "download_success",
    "download_status",
    "download_url",
    "download_path",
    "http_status",
    "error",
]


DMA_FILTER_COLUMNS = [
    "document_name",
    "pdf_filename",
    "file_name",
    "file_url",
    "dbcomp_document_url",
    "dbcomp_document_id",
    "raw_links_json",
    "raw_result_text",
]

DMA_EXCLUSION_RE = re.compile(
    r"(?i)(?:\bDMA\b|Digital[-_\s]+Markets[-_\s]+Act|Regulation[-_\s]*\(?EU\)?[-_\s]*2022[-_\s]*/[-_\s]*1925|2022[-_\s]*/[-_\s]*1925)"
)


def flag_dma_document_rows(df):
    """Return a boolean Series for rows that appear to concern the Digital Markets Act (DMA)."""
    if df.empty:
        return pd.Series(False, index=df.index)

    available_cols = [col for col in DMA_FILTER_COLUMNS if col in df.columns]
    if not available_cols:
        return pd.Series(False, index=df.index)

    combined = (
        df[available_cols]
        .fillna("")
        .astype(str)
        .agg(" ".join, axis=1)
    )
    return combined.str.contains(DMA_EXCLUSION_RE, na=False)


def remove_dma_document_rows(df, context="manifest"):
    """Remove DMA / Digital Markets Act rows when EXCLUDE_DMA_DOCUMENTS is enabled."""
    if not globals().get("EXCLUDE_DMA_DOCUMENTS", True):
        return df.copy()

    mask = flag_dma_document_rows(df)
    removed = int(mask.sum())
    if removed:
        print(f"DMA filter removed {removed:,} row(s) from {context}.")
        sample_cols = [col for col in ["case_number", "document_name", "pdf_filename", "file_name", "file_url"] if col in df.columns]
        display(df.loc[mask, sample_cols].head(20))
    return df.loc[~mask].copy()


def normalize_dbcomp_case_number(value):
    """Normalize DB-COMP case numbers lightly while preserving the source format."""
    value = clean_text(value)
    if not value or value.lower() in {"nan", "none", "<na>"}:
        return ""
    value = value.replace("_", ".")
    value = re.sub(r"\s+", "", value)
    value = re.sub(r"^[Cc][Oo][Mm][Pp][./_-]?", "COMP/", value)
    value = re.sub(r"^[Aa][Tt][./_-]?", "AT.", value)
    return value.strip(" .;,")


def extract_dbcomp_case_numbers_from_filename_or_text(row):
    """
    Extract DB-COMP case numbers from the beginning of PDF/file names.

    Examples:
        37919-37391-Bank-charges...pdf -> 37919;37391
        399904-rechargeable-battery.pdf -> 399904

    Falls back to raw text only if the filename does not contain a leading case-number block.
    """
    candidates = [
        row.get("pdf_filename", ""),
        row.get("file_name", ""),
        extract_filename_from_url(row.get("file_url", "")),
    ]

    for candidate in candidates:
        name = os.path.basename(clean_text(candidate))
        if not name:
            continue
        stem = re.sub(r"\.[A-Za-z0-9]{1,5}$", "", name)
        parts = re.split(r"[-_\s]+", stem)
        found = []
        for part in parts:
            token = normalize_dbcomp_case_number(part)
            if re.fullmatch(r"\d{2,6}", token) or re.fullmatch(r"(?:AT\.|COMP/)?\d{2,6}", token, flags=re.IGNORECASE):
                found.append(token)
                continue
            break
        if found:
            return ";".join(dict.fromkeys(found))

    # Conservative fallback: take only leading case-like tokens from raw result text.
    text = clean_text(row.get("raw_result_text", ""))
    m = re.match(r"^((?:\d{2,6})(?:[\s,;/+-]+\d{2,6})*)\b", text)
    if m:
        nums = re.findall(r"\d{2,6}", m.group(1))
        return ";".join(dict.fromkeys(nums))

    return ""


def extract_dbcomp_document_id_from_url(value):
    """Extract DB-COMP document ID from known DB-COMP file/document URLs."""
    value = clean_text(value)
    if not value:
        return ""

    patterns = [
        r"document_(\d+)\.download",
        r"document[_/-]?(\d+)",
        r"[?&]document(?:_id|Id|id)?=(\d+)",
        r"[?&]id=(\d+)",
    ]
    for pattern in patterns:
        m = re.search(pattern, value, flags=re.IGNORECASE)
        if m:
            return m.group(1)
    return ""


def fill_missing_dbcomp_document_ids(df, start_id=9000):
    """Fill missing DB-COMP document IDs deterministically after sorting by file_url.

    Existing IDs are preserved. Missing IDs are first extracted from URL-like columns.
    Any rows still missing an ID are ordered by file_url and assigned 9000, 9001, ... .
    """
    df = df.copy()
    if "dbcomp_document_id" not in df.columns:
        df["dbcomp_document_id"] = ""

    df["dbcomp_document_id"] = (
        df["dbcomp_document_id"]
        .fillna("")
        .astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .str.strip()
    )
    missing = df["dbcomp_document_id"].eq("") | df["dbcomp_document_id"].str.lower().isin({"nan", "none", "<na>"})

    for col in ["file_url", "dbcomp_document_url", "download_url"]:
        if col not in df.columns:
            continue
        extracted = df.loc[missing, col].map(extract_dbcomp_document_id_from_url)
        df.loc[missing, "dbcomp_document_id"] = extracted
        missing = df["dbcomp_document_id"].fillna("").astype(str).str.strip().eq("")

    if missing.any():
        used_numeric_ids = set(pd.to_numeric(df.loc[~missing, "dbcomp_document_id"], errors="coerce").dropna().astype(int).tolist())
        next_id = int(start_id)
        assignment_order = (
            df.loc[missing]
            .assign(_sort_file_url=df.loc[missing, "file_url"].fillna("").astype(str) if "file_url" in df.columns else "")
            .sort_values(["_sort_file_url"], kind="mergesort")
            .index
            .tolist()
        )
        for idx in assignment_order:
            while next_id in used_numeric_ids:
                next_id += 1
            df.at[idx, "dbcomp_document_id"] = str(next_id)
            used_numeric_ids.add(next_id)
            next_id += 1

    df["dbcomp_document_id"] = df["dbcomp_document_id"].fillna("").astype(str).str.strip()
    return df


def postprocess_db_comp_manifest(df):
    """Add DB-COMP case-number audit columns, fill document IDs, remove empty document_type metadata, and exclude DMA rows."""
    df = df.copy()
    df = remove_dma_document_rows(df, context="DB-COMP manifest")

    if "document_type" in df.columns:
        df = df.drop(columns=["document_type"])

    if "db_comp_case_number_saved" not in df.columns:
        if "case_number" in df.columns:
            df["db_comp_case_number_saved"] = df["case_number"].fillna("").map(normalize_dbcomp_case_number)
        else:
            df["db_comp_case_number_saved"] = ""

    df["db_comp_case_number_saved"] = df["db_comp_case_number_saved"].fillna("").map(normalize_dbcomp_case_number)
    df["db_comp_case_number_extracted_from_text"] = df.apply(extract_dbcomp_case_numbers_from_filename_or_text, axis=1)

    saved = df["db_comp_case_number_saved"].fillna("").astype(str).str.strip()
    extracted = df["db_comp_case_number_extracted_from_text"].fillna("").astype(str).str.strip()
    df["case_number"] = saved.where(saved.ne(""), extracted)

    df = fill_missing_dbcomp_document_ids(df, start_id=9000)

    ordered = [col for col in WEB_SCRAPE_COLUMNS if col in df.columns]
    remaining = [col for col in df.columns if col not in ordered]
    return df[ordered + remaining]


def save_dataframe(df, csv_path, parquet_path=None):
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    if parquet_path:
        parquet_df = df.copy()

        if "http_status" in parquet_df.columns:
            parquet_df["http_status"] = pd.to_numeric(
                parquet_df["http_status"].replace("", pd.NA),
                errors="coerce",
            ).astype("Int64")

        if "download_success" in parquet_df.columns:
            parquet_df["download_success"] = parquet_df["download_success"].astype("boolean")

        for col in parquet_df.columns:
            if parquet_df[col].dtype == object:
                parquet_df[col] = parquet_df[col].astype("string")

        try:
            parquet_df.to_parquet(parquet_path, index=False)
        except Exception as e:
            logger.warning("Could not save parquet file %s: %s", parquet_path, e)


def save_web_scrape_manifest(df):
    df = postprocess_db_comp_manifest(df)
    save_dataframe(df, DB_COMP_MANIFEST_CSV, DB_COMP_MANIFEST_PARQUET)


def save_download_manifest(df):
    save_dataframe(df, DOWNLOAD_MANIFEST_CSV, DOWNLOAD_MANIFEST_PARQUET)


def prepare_download_result_columns_for_updates(df):
    """
    Make mutable download-result columns safe for scalar updates.
    """
    for col in DOWNLOAD_RESULT_COLUMNS:
        if col not in df.columns:
            df[col] = pd.NA
        df[col] = df[col].astype("object")
    return df


## 6. Fetch and parse DB-COMP pages

In [7]:
def fetch_page(session: requests.Session, page: int) -> str:
    url = make_page_url(page)
    raw_path = os.path.join(COM_DB_COMP_HTML_DIR, f"page_{page:04d}.html")

    if os.path.exists(raw_path) and not OVERWRITE_RAW_HTML:
        logger.info("Using existing raw HTML for page %s: %s", page, raw_path)
        with open(raw_path, "r", encoding="utf-8") as f:
            return f.read()

    response = session.get(url, headers=HEADERS, timeout=30)
    logger.info("Fetched page=%s status=%s length=%s", page, response.status_code, len(response.text))
    response.raise_for_status()

    with open(raw_path, "w", encoding="utf-8") as f:
        f.write(response.text)

    return response.text


def parse_total_results(html: str):
    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text("\n", strip=True)
    m = re.search(r"(\d+)\s+results", text, flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    return None


def parse_last_page(html: str):
    soup = BeautifulSoup(html, "html.parser")

    for a in soup.find_all("a", href=True):
        label = clean_text(a.get_text(" ", strip=True)).lower()
        if label == "last":
            m = re.search(r"[?&]page=(\d+)", a["href"])
            if m:
                return int(m.group(1))

    # Fallback: scan all page= links and take the maximum.
    page_numbers = []
    for a in soup.find_all("a", href=True):
        m = re.search(r"[?&]page=(\d+)", a["href"])
        if m:
            page_numbers.append(int(m.group(1)))

    return max(page_numbers) if page_numbers else None


def parse_result_block(block, page: int, result_index: int) -> dict:
    page_url = make_page_url(page)
    text = clean_text(block.get_text(" ", strip=True))

    links = []
    for a in block.find_all("a", href=True):
        link_text = clean_text(a.get_text(" ", strip=True))
        href = urljoin(BASE_URL, a["href"])
        links.append({"text": link_text, "href": href})

    # First non-empty link is usually the document link/name.
    document_name = ""
    dbcomp_document_url = ""
    for link in links:
        if link["text"]:
            document_name = link["text"]
            dbcomp_document_url = link["href"]
            break

    # If no link text exists, fall back to first link URL.
    if not dbcomp_document_url and links:
        dbcomp_document_url = links[0]["href"]

    dbcomp_document_id = ""
    m_doc = re.search(r"document_(\d+)\.download", dbcomp_document_url)
    if m_doc:
        dbcomp_document_id = m_doc.group(1)

    decision_date = ""
    m_date = re.search(r"Date:\s*([0-9]{1,2}\s+[A-Za-z]{3}\s+[0-9]{4})", text)
    if m_date:
        decision_date = m_date.group(1)


    pdf_filename = ""
    m_pdf = re.search(r"([^\s]+\.pdf)", text, flags=re.IGNORECASE)
    if m_pdf:
        pdf_filename = safe_filename(m_pdf.group(1))

    case_number = ""
    m_case = re.match(r"^([A-Za-z]*\.?\s*\d+[A-Za-z0-9./-]*)\s+", text)
    if m_case:
        case_number = clean_text(m_case.group(1))

    file_url = dbcomp_document_url
    file_name = pdf_filename or extract_filename_from_url(file_url)
    if not file_name and dbcomp_document_id:
        file_name = f"dbcomp_document_{dbcomp_document_id}.pdf"
    elif not file_name:
        file_name = f"dbcomp_document_{hash_text(file_url or text)}.pdf"

    file_extension = os.path.splitext(file_name)[1].lower().replace(".", "")

    return {
        "source": "db-comp.eu",
        "page": page,
        "page_url": page_url,
        "result_index": result_index,
        "db_comp_case_number_saved": case_number,
        "db_comp_case_number_extracted_from_text": "",
        "case_number": case_number,
        "document_name": document_name,
        "decision_date": decision_date,
        "pdf_filename": pdf_filename,
        "dbcomp_document_id": dbcomp_document_id,
        "dbcomp_document_url": dbcomp_document_url,
        "file_url": file_url,
        "file_name": file_name,
        "file_extension": file_extension,
        "raw_links_json": json.dumps(links, ensure_ascii=False),
        "raw_result_text": text,
        "scrape_status": "parsed",
        "scrape_error": "",
    }


def parse_results(html: str, page: int) -> list[dict]:
    soup = BeautifulSoup(html, "html.parser")
    rows = []

    blocks = soup.select("div.result")

    # Robust fallback in case the HTML structure changes.
    if not blocks:
        blocks = [
            tag for tag in soup.find_all(["div", "li", "article", "section"])
            if "document_" in str(tag) and ".download" in str(tag)
        ]

    for i, block in enumerate(blocks, start=1):
        row = parse_result_block(block, page=page, result_index=i)
        if row["case_number"] or row["document_name"] or row["dbcomp_document_url"]:
            rows.append(row)

    return rows


## 7. Build or load web scrape manifest

In [8]:
def build_web_scrape_manifest():
    existing_path = None
    if os.path.exists(DB_COMP_MANIFEST_CSV) and not REBUILD_WEB_SCRAPE_MANIFEST:
        existing_path = DB_COMP_MANIFEST_CSV
    elif os.path.exists(LEGACY_WEB_SCRAPE_MANIFEST_CSV) and not REBUILD_WEB_SCRAPE_MANIFEST:
        existing_path = LEGACY_WEB_SCRAPE_MANIFEST_CSV

    if existing_path:
        print(f"Loading existing DB-COMP manifest: {existing_path}")
        df = pd.read_csv(existing_path, low_memory=False)
        df = postprocess_db_comp_manifest(df)
        save_web_scrape_manifest(df)
        print(f"Rows: {len(df):,}")
        print(f"Saved normalized DB-COMP manifest: {DB_COMP_MANIFEST_CSV}")
        return df

    print("Building DB-COMP manifest")
    session = requests.Session()

    html_1 = fetch_page(session, page=1)
    total_results = parse_total_results(html_1)
    last_page = parse_last_page(html_1) or FALLBACK_LAST_PAGE

    if PAGE_LIMIT is not None:
        last_page = min(last_page, PAGE_LIMIT)

    print("Total results:", total_results)
    print("Last page to scrape:", last_page)

    all_rows = []

    for page in tqdm(range(1, last_page + 1), desc="Scraping DB-COMP pages"):
        try:
            html = fetch_page(session, page=page)
            rows = parse_results(html, page=page)
            all_rows.extend(rows)
            logger.info("Parsed page=%s rows=%s", page, len(rows))
        except Exception as e:
            logger.exception("Failed page %s", page)
            all_rows.append({
                "source": "db-comp.eu",
                "page": page,
                "page_url": make_page_url(page),
                "result_index": pd.NA,
                "db_comp_case_number_saved": "",
                "db_comp_case_number_extracted_from_text": "",
                "case_number": "",
                "document_name": "",
                "decision_date": "",
                "pdf_filename": "",
                "dbcomp_document_id": "",
                "dbcomp_document_url": "",
                "file_url": "",
                "file_name": "",
                "file_extension": "",
                "raw_links_json": "[]",
                "raw_result_text": "",
                "scrape_status": "failed",
                "scrape_error": str(e),
            })

        sleep_random(REQUEST_SLEEP_SECONDS)

    df = pd.DataFrame(all_rows)
    df = postprocess_db_comp_manifest(df)
    save_web_scrape_manifest(df)
    print(f"Saved DB-COMP manifest: {DB_COMP_MANIFEST_CSV}")
    print(f"Rows: {len(df):,}")
    return df


db_comp_manifest = build_web_scrape_manifest()
web_scrape_manifest = db_comp_manifest  # backward-compatible alias for cells below

display(db_comp_manifest.head(10))
print("Rows:", len(db_comp_manifest))
print("Missing final case_number:", (db_comp_manifest["case_number"].fillna("").astype(str).str.strip() == "").sum())
print("Saved case number missing:", (db_comp_manifest["db_comp_case_number_saved"].fillna("").astype(str).str.strip() == "").sum())
print("Extracted from filename/text present:", (db_comp_manifest["db_comp_case_number_extracted_from_text"].fillna("").astype(str).str.strip() != "").sum())


Loading existing DB-COMP manifest: /home/edik/projects/eccjeu/output/com_db_comp/db_comp_manifest.csv
Rows: 833
Saved normalized DB-COMP manifest: /home/edik/projects/eccjeu/output/com_db_comp/db_comp_manifest.csv


,dbcomp_document_id,case_number,source,page,page_url,result_index,db_comp_case_number_saved,db_comp_case_number_extracted_from_text,document_name,decision_date,pdf_filename,dbcomp_document_url,file_url,file_name,file_extension,raw_links_json,raw_result_text,scrape_status,scrape_error
0,1061,93,db-comp.eu,1,https://db-comp.eu/?slotName=Slot_Main_3&Searc...,1,93,93,European Machine Tool Exhibitions (EEMO),13 Mar 1969,CELEX-31979D0037-EN-TXT.pdf,https://db-comp.eu/document_1061.download,https://db-comp.eu/document_1061.download,CELEX-31979D0037-EN-TXT.pdf,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",93 European Machine Tool Exhibitions (EEMO) CE...,parsed,NaN
1,1064,399904,db-comp.eu,1,https://db-comp.eu/?slotName=Slot_Main_3&Searc...,2,399904,399904,Rechargeable battery,12 Dec 2016,399904-rechargeable-battery-_Dec-2016_.pdf,https://db-comp.eu/document_1064.download,https://db-comp.eu/document_1064.download,399904-rechargeable-battery-_Dec-2016_.pdf,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",399904 Rechargeable battery 399904-rechargeabl...,parsed,NaN
2,1065,40481,db-comp.eu,1,https://db-comp.eu/?slotName=Slot_Main_3&Searc...,3,40481,40481,Occupant Safety Systems (II) supplied to the V...,5 Mar 2019,40481-Occupant-Safety-Systems.pdf,https://db-comp.eu/document_1065.download,https://db-comp.eu/document_1065.download,40481-Occupant-Safety-Systems.pdf,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",40481 Occupant Safety Systems (II) supplied to...,parsed,NaN
3,1066,40360,db-comp.eu,1,https://db-comp.eu/?slotName=Slot_Main_3&Searc...,4,40360,40360,Production and distribution of audiobooks,19 Jan 2017,40360-production-and-distribution-of-audiobook...,https://db-comp.eu/document_1066.download,https://db-comp.eu/document_1066.download,40360-production-and-distribution-of-audiobook...,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",40360 Production and distribution of audiobook...,parsed,NaN
4,1067,40291,db-comp.eu,1,https://db-comp.eu/?slotName=Slot_Main_3&Searc...,5,40291,40291,Aquatrend,21 Jan 2016,40291-Aquatrend-_Jan-2016_.pdf,https://db-comp.eu/document_1067.download,https://db-comp.eu/document_1067.download,40291-Aquatrend-_Jan-2016_.pdf,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",40291 Aquatrend 40291-Aquatrend-(Jan-2016).pdf...,parsed,NaN
5,1068,40208,db-comp.eu,1,https://db-comp.eu/?slotName=Slot_Main_3&Searc...,6,40208,40208,International Skating Unionâ€™s Eligibility rules,8 Dec 2017,40208-International-Skating-Union-s-Eligibilit...,https://db-comp.eu/document_1068.download,https://db-comp.eu/document_1068.download,40208-International-Skating-Union-s-Eligibilit...,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",40208 International Skating Unionâ€™s Eligibil...,parsed,NaN
6,1069,40169,db-comp.eu,1,https://db-comp.eu/?slotName=Slot_Main_3&Searc...,7,40169,40169,MACO,11 Mar 2016,40169-MACO-_March-2016_.pdf,https://db-comp.eu/document_1069.download,https://db-comp.eu/document_1069.download,40169-MACO-_March-2016_.pdf,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",40169 MACO 40169-MACO-(March-2016).pdf Agreeme...,parsed,NaN
7,1072,40113,db-comp.eu,1,https://db-comp.eu/?slotName=Slot_Main_3&Searc...,8,40113,40113,Spark Plugs,21 Feb 2018,40113-Spark-Plugs.pdf,https://db-comp.eu/document_1072.download,https://db-comp.eu/document_1072.download,40113-Spark-Plugs.pdf,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",40113 Spark Plugs 40113-Spark-Plugs.pdf Agreem...,parsed,NaN
8,1073,40105,db-comp.eu,1,https://db-comp.eu/?slotName=Slot_Main_3&Searc...,9,40105,40105,UEFA Financial Fair Play Rules,24 Oct 2014,40105-UEFA-Financial-Fair-Play-Rules-_Oct-2014...,https://db-comp.eu/document_1073.download,https://db-comp.eu/document_1073.download,40105-UEFA-Financial-Fair-Play-Rules-_Oct-2014...,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",40105 UEFA Financial Fair Play Rules 40105-UEF...,parsed,NaN

Rows: 833
Missing final case_number: 0
Saved case number missing: 48
Extracted from filename/text present: 833


## 8. Build or load download manifest

In [9]:
def make_download_key(row):
    if clean_text(row.get("dbcomp_document_id", "")):
        return f"dbcomp_document_{clean_text(row.get('dbcomp_document_id'))}"
    if clean_text(row.get("file_url", "")):
        return hash_text(clean_text(row.get("file_url")), n=16)
    return hash_text(str(row.to_dict()), n=16)


def create_base_download_manifest(web_df):
    df = web_df.copy()
    df = remove_dma_document_rows(df, context="download manifest")

    # Keep only rows with a file URL.
    df = df[df["file_url"].fillna("").astype(str).str.len() > 0].copy()

    df["download_key"] = df.apply(make_download_key, axis=1)
    df["download_success"] = pd.NA
    df["download_status"] = "pending"
    df["download_url"] = df["file_url"]
    df["download_path"] = pd.NA
    df["http_status"] = pd.NA
    df["error"] = pd.NA

    keep_cols = [
        "dbcomp_document_id",
        "case_number",
        "download_key",
        "source",
        "page",
        "document_name",
        "decision_date",
        "file_url",
        "file_name",
        "file_extension",
        *DOWNLOAD_RESULT_COLUMNS,
    ]

    for col in keep_cols:
        if col not in df.columns:
            df[col] = pd.NA

    df = df[keep_cols]
    df = fill_missing_dbcomp_document_ids(df, start_id=9000)
    df = df.drop_duplicates(subset=["download_key"], keep="first").reset_index(drop=True)
    df = prepare_download_result_columns_for_updates(df)
    return df


def merge_existing_download_progress(base_df, existing_df):
    """Preserve previous download statuses/results by download_key."""
    result_cols = DOWNLOAD_RESULT_COLUMNS

    existing_small = existing_df[["download_key", *[c for c in result_cols if c in existing_df.columns]]].copy()
    existing_small = existing_small.drop_duplicates(subset=["download_key"], keep="last")

    merged = base_df.drop(columns=[c for c in result_cols if c in base_df.columns], errors="ignore").merge(
        existing_small,
        on="download_key",
        how="left",
    )

    for col in result_cols:
        if col not in merged.columns:
            merged[col] = pd.NA

    merged["download_status"] = merged["download_status"].fillna("pending")
    merged = prepare_download_result_columns_for_updates(merged)
    return merged


def build_or_load_download_manifest(web_df):
    base = create_base_download_manifest(web_df)

    existing_download_manifest_path = None
    if os.path.exists(DOWNLOAD_MANIFEST_CSV):
        existing_download_manifest_path = DOWNLOAD_MANIFEST_CSV
    elif os.path.exists(LEGACY_DOWNLOAD_MANIFEST_CSV):
        existing_download_manifest_path = LEGACY_DOWNLOAD_MANIFEST_CSV

    if existing_download_manifest_path:
        print(f"Loading existing download manifest and preserving progress: {existing_download_manifest_path}")
        existing = pd.read_csv(existing_download_manifest_path, low_memory=False)
        existing = fill_missing_dbcomp_document_ids(existing, start_id=9000)

        if "download_key" not in existing.columns:
            print("Existing download manifest has no download_key; creating a fresh manifest from web manifest.")
            download_df = base
        else:
            download_df = merge_existing_download_progress(base, existing)
    else:
        print("Creating new download manifest")
        download_df = base

    download_df = prepare_download_result_columns_for_updates(download_df)
    save_download_manifest(download_df)

    print(f"Download manifest rows: {len(download_df):,}")
    print(download_df["download_status"].fillna("MISSING").value_counts(dropna=False))
    print("Saved:", DOWNLOAD_MANIFEST_CSV)

    return download_df


download_manifest = build_or_load_download_manifest(web_scrape_manifest)
download_manifest.head(10)


Loading existing download manifest and preserving progress: /home/edik/projects/eccjeu/output/com_db_comp/download_manifest.csv
Download manifest rows: 833
download_status
pending    833
Name: count, dtype: int64
Saved: /home/edik/projects/eccjeu/output/com_db_comp/db_comp_download_manifest.csv


,dbcomp_document_id,case_number,download_key,source,page,document_name,decision_date,file_url,file_name,file_extension,download_success,download_status,download_url,download_path,http_status,error
0,1061,93,dbcomp_document_1061,db-comp.eu,1,European Machine Tool Exhibitions (EEMO),13 Mar 1969,https://db-comp.eu/document_1061.download,CELEX-31979D0037-EN-TXT.pdf,pdf,NaN,pending,NaN,NaN,NaN,NaN
1,1064,399904,dbcomp_document_1064,db-comp.eu,1,Rechargeable battery,12 Dec 2016,https://db-comp.eu/document_1064.download,399904-rechargeable-battery-_Dec-2016_.pdf,pdf,NaN,pending,NaN,NaN,NaN,NaN
2,1065,40481,dbcomp_document_1065,db-comp.eu,1,Occupant Safety Systems (II) supplied to the V...,5 Mar 2019,https://db-comp.eu/document_1065.download,40481-Occupant-Safety-Systems.pdf,pdf,NaN,pending,NaN,NaN,NaN,NaN
3,1066,40360,dbcomp_document_1066,db-comp.eu,1,Production and distribution of audiobooks,19 Jan 2017,https://db-comp.eu/document_1066.download,40360-production-and-distribution-of-audiobook...,pdf,NaN,pending,NaN,NaN,NaN,NaN
4,1067,40291,dbcomp_document_1067,db-comp.eu,1,Aquatrend,21 Jan 2016,https://db-comp.eu/document_1067.download,40291-Aquatrend-_Jan-2016_.pdf,pdf,NaN,pending,NaN,NaN,NaN,NaN
5,1068,40208,dbcomp_document_1068,db-comp.eu,1,International Skating Unionâ€™s Eligibility rules,8 Dec 2017,https://db-comp.eu/document_1068.download,40208-International-Skating-Union-s-Eligibilit...,pdf,NaN,pending,NaN,NaN,NaN,NaN
6,1069,40169,dbcomp_document_1069,db-comp.eu,1,MACO,11 Mar 2016,https://db-comp.eu/document_1069.download,40169-MACO-_March-2016_.pdf,pdf,NaN,pending,NaN,NaN,NaN,NaN
7,1072,40113,dbcomp_document_1072,db-comp.eu,1,Spark Plugs,21 Feb 2018,https://db-comp.eu/document_1072.download,40113-Spark-Plugs.pdf,pdf,NaN,pending,NaN,NaN,NaN,NaN
8,1073,40105,dbcomp_document_1073,db-comp.eu,1,UEFA Financial Fair Play Rules,24 Oct 2014,https://db-comp.eu/document_1073.download,40105-UEFA-Financial-Fair-Play-Rules-_Oct-2014...,pdf,NaN,pending,NaN,NaN,NaN,NaN
9,1074,40098,dbcomp_document_1074,db-comp.eu,1,Blocktrains,15 Jul 2015,https://db-comp.eu/document_1074.download,40098-Blocktrains-_July-2015_.pdf,pdf,NaN,pending,NaN,NaN,NaN,NaN


## 9. Optional file downloader

By default, `DOWNLOAD_FILES = False`, so this section will only prepare the manifest. Set `DOWNLOAD_FILES = True` in the configuration cell when you want to download files.


In [10]:
def get_download_output_path(row):
    """Return the original DB-COMP source/download path without renaming."""
    file_name = clean_text(row.get("file_name", ""))

    if not file_name:
        ext = clean_text(row.get("file_extension", "")) or "pdf"
        file_name = f"{row.get('download_key', hash_text(row.get('file_url', '')))}.{ext}"

    file_name = safe_filename(file_name, default=f"{row.get('download_key', 'dbcomp_file')}.pdf")
    return os.path.join(COM_DB_COMP_FILES_DIR, file_name)


def download_one_file(session, row, overwrite=False):
    url = clean_text(row.get("file_url", ""))
    output_path = get_download_output_path(row)

    if not url:
        return {
            "download_success": False,
            "download_status": "failed_no_url",
            "download_url": url,
            "download_path": pd.NA,
            "http_status": pd.NA,
            "error": "Missing file_url",
        }

    if os.path.exists(output_path) and os.path.getsize(output_path) > 0 and not overwrite:
        return {
            "download_success": True,
            "download_status": "already_downloaded",
            "download_url": url,
            "download_path": output_path,
            "http_status": pd.NA,
            "error": pd.NA,
        }

    try:
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        response = session.get(url, headers=HEADERS, timeout=DOWNLOAD_TIMEOUT_SECONDS)
        status_code = response.status_code

        if status_code != 200 or not response.content:
            return {
                "download_success": False,
                "download_status": f"failed_http_{status_code}",
                "download_url": url,
                "download_path": pd.NA,
                "http_status": status_code,
                "error": f"HTTP {status_code}; length={len(response.content)}",
            }

        with open(output_path, "wb") as f:
            f.write(response.content)

        return {
            "download_success": True,
            "download_status": "downloaded",
            "download_url": url,
            "download_path": output_path,
            "http_status": status_code,
            "error": pd.NA,
        }

    except Exception as e:
        logger.exception("Failed to download %s", url)
        return {
            "download_success": False,
            "download_status": "failed_all_attempts",
            "download_url": url,
            "download_path": pd.NA,
            "http_status": pd.NA,
            "error": str(e),
        }


def run_downloads(download_df, limit=None, overwrite=False, save_every=10):
    if os.path.exists(DOWNLOAD_MANIFEST_CSV):
        df = pd.read_csv(DOWNLOAD_MANIFEST_CSV, low_memory=False)
    elif os.path.exists(LEGACY_DOWNLOAD_MANIFEST_CSV):
        df = pd.read_csv(LEGACY_DOWNLOAD_MANIFEST_CSV, low_memory=False)
    else:
        df = download_df.copy()

    df = fill_missing_dbcomp_document_ids(df, start_id=9000)

    df = prepare_download_result_columns_for_updates(df)

    status = df["download_status"].astype("string").str.strip().str.lower().fillna("pending")
    candidate_mask = ~status.isin(["already_downloaded", "downloaded"])
    candidate_indices = df.loc[candidate_mask].index.tolist()

    if limit is not None:
        candidate_indices = candidate_indices[:limit]

    print(f"Checking/downloading {len(candidate_indices):,} DB-COMP files")

    session = requests.Session()

    for counter, idx in enumerate(tqdm(candidate_indices, desc="Checking/downloading DB-COMP files"), start=1):
        row = df.loc[idx]
        result = download_one_file(session, row, overwrite=overwrite)

        for key, value in result.items():
            if key in df.columns:
                df.at[idx, key] = value

        if counter % save_every == 0:
            save_download_manifest(df)

        if result.get("download_status") != "already_downloaded":
            sleep_random(DOWNLOAD_SLEEP_SECONDS)

    save_download_manifest(df)
    print("Done.")
    print(df["download_status"].fillna("MISSING").value_counts(dropna=False))
    return df


In [11]:
if DOWNLOAD_FILES:
    download_manifest = run_downloads(
        download_manifest,
        limit=DOWNLOAD_LIMIT,
        overwrite=OVERWRITE_EXISTING_FILES,
        save_every=SAVE_EVERY,
    )
else:
    print("DOWNLOAD_FILES is False. No files downloaded.")
    print("Set DOWNLOAD_FILES = True in the config cell when you are ready.")

print(download_manifest["download_status"].fillna("MISSING").value_counts(dropna=False))
download_manifest.head(10)


Checking/downloading 833 DB-COMP files


Checking/downloading DB-COMP files:   0%|          | 0/833 [00:00<?, ?it/s]

Done.
download_status
downloaded            826
already_downloaded      7
Name: count, dtype: int64
download_status
downloaded            826
already_downloaded      7
Name: count, dtype: int64


,dbcomp_document_id,case_number,download_key,source,page,document_name,decision_date,file_url,file_name,file_extension,download_success,download_status,download_url,download_path,http_status,error
0,1061,93,dbcomp_document_1061,db-comp.eu,1,European Machine Tool Exhibitions (EEMO),13 Mar 1969,https://db-comp.eu/document_1061.download,CELEX-31979D0037-EN-TXT.pdf,pdf,True,already_downloaded,https://db-comp.eu/document_1061.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,<NA>,<NA>
1,1064,399904,dbcomp_document_1064,db-comp.eu,1,Rechargeable battery,12 Dec 2016,https://db-comp.eu/document_1064.download,399904-rechargeable-battery-_Dec-2016_.pdf,pdf,True,downloaded,https://db-comp.eu/document_1064.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,200,<NA>
2,1065,40481,dbcomp_document_1065,db-comp.eu,1,Occupant Safety Systems (II) supplied to the V...,5 Mar 2019,https://db-comp.eu/document_1065.download,40481-Occupant-Safety-Systems.pdf,pdf,True,downloaded,https://db-comp.eu/document_1065.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,200,<NA>
3,1066,40360,dbcomp_document_1066,db-comp.eu,1,Production and distribution of audiobooks,19 Jan 2017,https://db-comp.eu/document_1066.download,40360-production-and-distribution-of-audiobook...,pdf,True,downloaded,https://db-comp.eu/document_1066.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,200,<NA>
4,1067,40291,dbcomp_document_1067,db-comp.eu,1,Aquatrend,21 Jan 2016,https://db-comp.eu/document_1067.download,40291-Aquatrend-_Jan-2016_.pdf,pdf,True,downloaded,https://db-comp.eu/document_1067.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,200,<NA>
5,1068,40208,dbcomp_document_1068,db-comp.eu,1,International Skating Unionâ€™s Eligibility rules,8 Dec 2017,https://db-comp.eu/document_1068.download,40208-International-Skating-Union-s-Eligibilit...,pdf,True,downloaded,https://db-comp.eu/document_1068.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,200,<NA>
6,1069,40169,dbcomp_document_1069,db-comp.eu,1,MACO,11 Mar 2016,https://db-comp.eu/document_1069.download,40169-MACO-_March-2016_.pdf,pdf,True,downloaded,https://db-comp.eu/document_1069.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,200,<NA>
7,1072,40113,dbcomp_document_1072,db-comp.eu,1,Spark Plugs,21 Feb 2018,https://db-comp.eu/document_1072.download,40113-Spark-Plugs.pdf,pdf,True,downloaded,https://db-comp.eu/document_1072.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,200,<NA>
8,1073,40105,dbcomp_document_1073,db-comp.eu,1,UEFA Financial Fair Play Rules,24 Oct 2014,https://db-comp.eu/document_1073.download,40105-UEFA-Financial-Fair-Play-Rules-_Oct-2014...,pdf,True,downloaded,https://db-comp.eu/document_1073.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,200,<NA>
9,1074,40098,dbcomp_document_1074,db-comp.eu,1,Blocktrains,15 Jul 2015,https://db-comp.eu/document_1074.download,40098-Blocktrains-_July-2015_.pdf,pdf,True,downloaded,https://db-comp.eu/document_1074.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,200,<NA>


## 10. Quick checks

In [12]:
print("DB-COMP manifest:", DB_COMP_MANIFEST_CSV)
print("Download manifest:", DOWNLOAD_MANIFEST_CSV)
print("Raw HTML folder:", COM_DB_COMP_HTML_DIR)
print("Raw files folder:", COM_DB_COMP_FILES_DIR)

print("DB-COMP rows:", len(db_comp_manifest))
print("Download rows:", len(download_manifest))

print("DMA rows remaining in DB-COMP manifest:", int(flag_dma_document_rows(db_comp_manifest).sum()))
print("DMA rows remaining in download manifest:", int(flag_dma_document_rows(download_manifest).sum()))

print("Missing saved case number:", (db_comp_manifest["db_comp_case_number_saved"].fillna("").astype(str).str.strip() == "").sum())
print("Missing final case_number:", (db_comp_manifest["case_number"].fillna("").astype(str).str.strip() == "").sum())

display(db_comp_manifest.head(20))


DB-COMP manifest: /home/edik/projects/eccjeu/output/com_db_comp/db_comp_manifest.csv
Download manifest: /home/edik/projects/eccjeu/output/com_db_comp/db_comp_download_manifest.csv
Raw HTML folder: /home/edik/projects/eccjeu/data/raw/com_db_comp/html
Raw files folder: /home/edik/projects/eccjeu/data/raw/com_db_comp/files
DB-COMP rows: 833
Download rows: 833
DMA rows remaining in DB-COMP manifest: 0
DMA rows remaining in download manifest: 0
Missing saved case number: 48
Missing final case_number: 0


,dbcomp_document_id,case_number,source,page,page_url,result_index,db_comp_case_number_saved,db_comp_case_number_extracted_from_text,document_name,decision_date,pdf_filename,dbcomp_document_url,file_url,file_name,file_extension,raw_links_json,raw_result_text,scrape_status,scrape_error
0,1061,93,db-comp.eu,1,https://db-comp.eu/?slotName=Slot_Main_3&Searc...,1,93,93,European Machine Tool Exhibitions (EEMO),13 Mar 1969,CELEX-31979D0037-EN-TXT.pdf,https://db-comp.eu/document_1061.download,https://db-comp.eu/document_1061.download,CELEX-31979D0037-EN-TXT.pdf,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",93 European Machine Tool Exhibitions (EEMO) CE...,parsed,NaN
1,1064,399904,db-comp.eu,1,https://db-comp.eu/?slotName=Slot_Main_3&Searc...,2,399904,399904,Rechargeable battery,12 Dec 2016,399904-rechargeable-battery-_Dec-2016_.pdf,https://db-comp.eu/document_1064.download,https://db-comp.eu/document_1064.download,399904-rechargeable-battery-_Dec-2016_.pdf,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",399904 Rechargeable battery 399904-rechargeabl...,parsed,NaN
2,1065,40481,db-comp.eu,1,https://db-comp.eu/?slotName=Slot_Main_3&Searc...,3,40481,40481,Occupant Safety Systems (II) supplied to the V...,5 Mar 2019,40481-Occupant-Safety-Systems.pdf,https://db-comp.eu/document_1065.download,https://db-comp.eu/document_1065.download,40481-Occupant-Safety-Systems.pdf,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",40481 Occupant Safety Systems (II) supplied to...,parsed,NaN
3,1066,40360,db-comp.eu,1,https://db-comp.eu/?slotName=Slot_Main_3&Searc...,4,40360,40360,Production and distribution of audiobooks,19 Jan 2017,40360-production-and-distribution-of-audiobook...,https://db-comp.eu/document_1066.download,https://db-comp.eu/document_1066.download,40360-production-and-distribution-of-audiobook...,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",40360 Production and distribution of audiobook...,parsed,NaN
4,1067,40291,db-comp.eu,1,https://db-comp.eu/?slotName=Slot_Main_3&Searc...,5,40291,40291,Aquatrend,21 Jan 2016,40291-Aquatrend-_Jan-2016_.pdf,https://db-comp.eu/document_1067.download,https://db-comp.eu/document_1067.download,40291-Aquatrend-_Jan-2016_.pdf,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",40291 Aquatrend 40291-Aquatrend-(Jan-2016).pdf...,parsed,NaN
5,1068,40208,db-comp.eu,1,https://db-comp.eu/?slotName=Slot_Main_3&Searc...,6,40208,40208,International Skating Unionâ€™s Eligibility rules,8 Dec 2017,40208-International-Skating-Union-s-Eligibilit...,https://db-comp.eu/document_1068.download,https://db-comp.eu/document_1068.download,40208-International-Skating-Union-s-Eligibilit...,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",40208 International Skating Unionâ€™s Eligibil...,parsed,NaN
6,1069,40169,db-comp.eu,1,https://db-comp.eu/?slotName=Slot_Main_3&Searc...,7,40169,40169,MACO,11 Mar 2016,40169-MACO-_March-2016_.pdf,https://db-comp.eu/document_1069.download,https://db-comp.eu/document_1069.download,40169-MACO-_March-2016_.pdf,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",40169 MACO 40169-MACO-(March-2016).pdf Agreeme...,parsed,NaN
7,1072,40113,db-comp.eu,1,https://db-comp.eu/?slotName=Slot_Main_3&Searc...,8,40113,40113,Spark Plugs,21 Feb 2018,40113-Spark-Plugs.pdf,https://db-comp.eu/document_1072.download,https://db-comp.eu/document_1072.download,40113-Spark-Plugs.pdf,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",40113 Spark Plugs 40113-Spark-Plugs.pdf Agreem...,parsed,NaN
8,1073,40105,db-comp.eu,1,https://db-comp.eu/?slotName=Slot_Main_3&Searc...,9,40105,40105,UEFA Financial Fair Play Rules,24 Oct 2014,40105-UEFA-Financial-Fair-Play-Rules-_Oct-2014...,https://db-comp.eu/document_1073.download,https://db-comp.eu/document_1073.download,40105-UEFA-Financial-Fair-Play-Rules-_Oct-2014...,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",40105 UEFA Financial Fair Play Rules 40105-UEF...,parsed,NaN